In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

# =========================
# 환경설정
# =========================
load_dotenv("key.env")
SERVICE_KEY = os.getenv("SERVICE_KEY")

BASE_URL = "https://apis.data.go.kr/B551011/KorService2/areaBasedList2"
OUTPUT_DIR = "data/raw"

if not SERVICE_KEY:
    raise ValueError("SERVICE_KEY가 없습니다. key.env 파일을 확인하세요.")

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# 공통 함수
# =========================
def safe_float(value):
    if value in (None, "", "None"):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def request_page(content_type_id: int, area_code: int = 1, page_no: int = 1, num_of_rows: int = 100):
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_of_rows,
        "pageNo": page_no,
        "MobileOS": "ETC",
        "MobileApp": "MedicalTourPlanner",
        "_type": "json",
        "contentTypeId": content_type_id,
        "areaCode": area_code,   # 서울 = 1
        "arrange": "A"
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        body = data.get("response", {}).get("body", {})
        items = body.get("items", {}).get("item", [])
        total_count = body.get("totalCount", 0)

        if isinstance(items, dict):
            items = [items]

        return {
            "success": True,
            "items": items,
            "totalCount": total_count
        }

    except Exception as e:
        print(f"❌ 오류: {e}")
        return {
            "success": False,
            "items": [],
            "totalCount": 0
        }


def normalize_item(item, content_type_name: str):
    return {
        "contentid": item.get("contentid"),
        "contenttypeid": item.get("contenttypeid"),
        "content_type_name": content_type_name,
        "name": item.get("title", ""),
        "address": item.get("addr1", ""),
        "address_detail": item.get("addr2", ""),
        "latitude": safe_float(item.get("mapy")),
        "longitude": safe_float(item.get("mapx")),
        "tel": item.get("tel", ""),
        "zipcode": item.get("zipcode", ""),
        "first_image": item.get("firstimage", ""),
        "first_image2": item.get("firstimage2", ""),
        "mlevel": item.get("mlevel", ""),
        "created_time": item.get("createdtime", ""),
        "modified_time": item.get("modifiedtime", ""),
        "lDongRegnCd": item.get("lDongRegnCd", ""),
        "lDongSignguCd": item.get("lDongSignguCd", ""),
        "lclsSystm1": item.get("lclsSystm1", ""),
        "lclsSystm2": item.get("lclsSystm2", ""),
        "lclsSystm3": item.get("lclsSystm3", "")
    }


def collect_all_by_type(content_type_id: int, content_type_name: str, area_code: int = 1):
    all_data = []
    page_no = 1
    total_count = None

    print("=" * 70)
    print(f"📥 서울 {content_type_name} 수집 시작")
    print("=" * 70)

    while True:
        print(f"🔍 {content_type_name} 페이지 {page_no} 수집 중...", end=" ")

        result = request_page(
            content_type_id=content_type_id,
            area_code=area_code,
            page_no=page_no,
            num_of_rows=100
        )

        if not result["success"]:
            print("❌ 실패")
            break

        items = result["items"]

        if not items:
            print("⚠️ 데이터 없음")
            break

        if total_count is None:
            total_count = result["totalCount"]
            print(f"\n   📊 전체 {total_count}개 발견")

        for item in items:
            all_data.append(normalize_item(item, content_type_name))

        print(f"✅ {len(items)}개 (누적: {len(all_data)}/{total_count})")

        if len(all_data) >= total_count:
            print(f"\n✅ {content_type_name} 전체 수집 완료")
            break

        page_no += 1
        time.sleep(0.3)

    df = pd.DataFrame(all_data)

    if not df.empty:
        df = df.drop_duplicates(subset=["contentid"], keep="first")

    return df


def save_xlsx(df: pd.DataFrame, filename: str):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_excel(path, index=False)
    print(f"💾 저장 완료: {path}")


def print_basic_stats(df: pd.DataFrame, title: str):
    print("=" * 70)
    print(f"📊 {title} 통계")
    print("=" * 70)
    print(f"총 개수: {len(df):,}개")

    if df.empty:
        return

    print("\nlclsSystm2 분포:")
    print(df["lclsSystm2"].value_counts(dropna=False).head(10))

    print("\nlclsSystm3 분포:")
    print(df["lclsSystm3"].value_counts(dropna=False).head(15))


# =========================
# 후처리
# =========================
def split_cafe_from_restaurants(df_restaurants: pd.DataFrame):
    """
    음식점에서 카페 분리
    포함:
    - 중분류 FD05 전체 (카페/찻집 계열)
    - 소분류 FD030100 (제과)
    제외:
    - 나머지 음식점
    """
    cafe_condition = (
        (df_restaurants["lclsSystm2"] == "FD05") |
        (df_restaurants["lclsSystm3"] == "FD030100")
    )

    cafes = df_restaurants[cafe_condition].copy()
    restaurants_no_cafe = df_restaurants[~cafe_condition].copy()

    return cafes, restaurants_no_cafe


def remove_medical_from_shopping(df_shopping: pd.DataFrame):
    """
    쇼핑 데이터에서 의료시설명 제거
    """
    medical_keywords = [
        "병원", "의원", "한의원", "클리닉",
        "치과", "성형", "피부과", "안과",
        "내과", "외과", "정형외과", "산부인과",
        "소아과", "이비인후과", "비뇨기과",
        "신경외과", "재활의학과", "마취통증의학과"
    ]

    medical_pattern = "|".join(medical_keywords)
    medical_mask = df_shopping["name"].astype(str).str.contains(medical_pattern, case=False, na=False)

    shopping_clean = df_shopping[~medical_mask].copy()
    medical_removed = df_shopping[medical_mask].copy()

    return shopping_clean, medical_removed


# =========================
# 메인
# =========================
def main():
    type_map = {
        15: "행사공연축제",
        39: "음식점",
        32: "숙박",
        38: "쇼핑",
        14: "문화시설",
        28: "레포츠",
        12: "관광지"
    }

    results = {}

    # 1) 원본 수집
    for content_type_id, content_type_name in type_map.items():
        df = collect_all_by_type(content_type_id, content_type_name, area_code=1)
        results[content_type_name] = df

        if df.empty:
            print(f"❌ {content_type_name} 데이터 없음")
            continue

        print_basic_stats(df, content_type_name)

    print("\n" + "=" * 70)
    print("💾 타입별 파일 저장")
    print("=" * 70)

    # 2) 행사공연축제
    if not results["행사공연축제"].empty:
        save_xlsx(results["행사공연축제"], "seoul_festival_event_tourapi.xlsx")

    # 3) 음식점 -> 카페 / 카페제외 분리
    if not results["음식점"].empty:
        df_restaurants = results["음식점"]
        cafes, restaurants_no_cafe = split_cafe_from_restaurants(df_restaurants)

        save_xlsx(restaurants_no_cafe, "seoul_restaurants_no_cafe_tourapi.xlsx")
        save_xlsx(cafes, "seoul_cafes_tourapi.xlsx")

    # 4) 숙박
    if not results["숙박"].empty:
        save_xlsx(results["숙박"], "seoul_stay_tourapi.xlsx")

    # 5) 쇼핑 -> 의료 제거 버전
    if not results["쇼핑"].empty:
        shopping_clean, medical_removed = remove_medical_from_shopping(results["쇼핑"])

        save_xlsx(shopping_clean, "seoul_shopping_no_medical_tourapi.xlsx")
        # save_xlsx(medical_removed, "seoul_shopping_removed_medical_reference.xlsx")

    # 6) 문화시설
    if not results["문화시설"].empty:
        save_xlsx(results["문화시설"], "seoul_cultural_facilities_tourapi.xlsx")

    # 7) 레포츠
    if not results["레포츠"].empty:
        save_xlsx(results["레포츠"], "seoul_leports_tourapi.xlsx")

    # 8) 관광지
    if not results["관광지"].empty:
        save_xlsx(results["관광지"], "seoul_tourist_spots_tourapi.xlsx")

    print("\n" + "=" * 70)
    print("✅ 최종 완료 파일")
    print("=" * 70)
    print("1. seoul_festival_event_tourapi.xlsx")
    print("2. seoul_restaurants_no_cafe_tourapi.xlsx")
    print("3. seoul_cafes_tourapi.xlsx   ← FD05 + FD030100(제과) 포함")
    print("4. seoul_stay_tourapi.xlsx")
    print("5. seoul_shopping_no_medical_tourapi.xlsx")
    print("6. seoul_cultural_facilities_tourapi.xlsx")
    print("7. seoul_leports_tourapi.xlsx")
    print("8. seoul_tourist_spots_tourapi.xlsx")
    print("=" * 70)


if __name__ == "__main__":
    main()

📥 서울 행사공연축제 수집 시작
🔍 행사공연축제 페이지 1 수집 중... 
   📊 전체 177개 발견
✅ 100개 (누적: 100/177)
🔍 행사공연축제 페이지 2 수집 중... ✅ 77개 (누적: 177/177)

✅ 행사공연축제 전체 수집 완료
📊 행사공연축제 통계
총 개수: 177개

lclsSystm2 분포:
lclsSystm2
EV01    105
EV03     54
EV02     18
Name: count, dtype: int64

lclsSystm3 분포:
lclsSystm3
EV010200    52
EV030400    43
EV010600    38
EV010400    13
EV030200     6
EV020700     5
EV030100     4
EV020800     3
EV020200     3
EV020500     3
EV010500     2
EV020900     2
EV020100     1
EV030300     1
EV020300     1
Name: count, dtype: int64
📥 서울 음식점 수집 시작
🔍 음식점 페이지 1 수집 중... 
   📊 전체 1222개 발견
✅ 100개 (누적: 100/1222)
🔍 음식점 페이지 2 수집 중... ✅ 100개 (누적: 200/1222)
🔍 음식점 페이지 3 수집 중... ✅ 100개 (누적: 300/1222)
🔍 음식점 페이지 4 수집 중... ✅ 100개 (누적: 400/1222)
🔍 음식점 페이지 5 수집 중... ✅ 100개 (누적: 500/1222)
🔍 음식점 페이지 6 수집 중... ✅ 100개 (누적: 600/1222)
🔍 음식점 페이지 7 수집 중... ✅ 100개 (누적: 700/1222)
🔍 음식점 페이지 8 수집 중... ✅ 100개 (누적: 800/1222)
🔍 음식점 페이지 9 수집 중... ✅ 100개 (누적: 900/1222)
🔍 음식점 페이지 10 수집 중... ✅ 100개 (누적: 1000/1222)
🔍 음식점 페이지 11 수